<a href="https://colab.research.google.com/github/bhavyaaggarwal8984/NIC-training/blob/main/NICLabExercises_Bhavya_Aggarwal.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Submitted By:** Bhavya Aggarwal | **Designation:** Scientific Officer/Engineer-SB & DIO, NIC

# GOT-A Practical Lab Exercises
**Program:** Government Officer Training – Advanced in Artificial Intelligence (GOT-A)  
**Exercises Covered:** 1 to 5

### Environment Notes
- Designed to run sequentially from top to bottom in Google Colab or Jupyter Notebook.
- Generated Excel sheets and images are stored in the `lab_outputs` folder.
- Exercise 5 starts a Gradio web interface (`share=True` generates a temporary public link in Colab).

## Setup & Library Imports

In [ ]:
%pip install -q pandas numpy matplotlib openpyxl opencv-python scikit-image wordcloud gradio kagglehub

In [ ]:
import os
import re
import glob
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Image as IPyImage, Markdown

warnings.filterwarnings("ignore")
OUTPUT_DIR = "lab_outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)
print("Working output directory:", os.path.abspath(OUTPUT_DIR))

# Exercise 1: Data Exploration & Excel Export with Pandas

**Task:** Create a DataFrame with numerical columns `f1`–`f4`, plot `f2` against `f3`, compute the row-wise mean in `f5`, and export all five columns to an Excel file.

In [ ]:
# Generate reproducible numerical data
rng = np.random.default_rng(108)
num_samples = 100

metrics_df = pd.DataFrame({
    "f1": rng.uniform(10, 100, num_samples),
    "f2": rng.uniform(10, 100, num_samples),
    "f3": rng.uniform(10, 100, num_samples),
    "f4": rng.uniform(10, 100, num_samples),
})

# Scatter plot of f2 vs f3
fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(metrics_df["f2"], metrics_df["f3"], color="#1f77b4", edgecolors="white", s=55, alpha=0.8)
ax.set_xlabel("Feature f2")
ax.set_ylabel("Feature f3")
ax.set_title("f2 vs f3 Distribution")
ax.grid(True, linestyle=":", alpha=0.5)
plt.tight_layout()
plt.show()

# Calculate row-wise mean as f5
metrics_df["f5"] = metrics_df[["f1", "f2", "f3", "f4"]].mean(axis=1)

print("Shape:", metrics_df.shape)
display(metrics_df.head())
display(metrics_df.describe().round(2))

# Save all columns to Excel
report_file = os.path.join(OUTPUT_DIR, "Ex1_Metrics_Summary.xlsx")
metrics_df.to_excel(report_file, index=False)
print("Saved Excel report to:", os.path.abspath(report_file))

# Exercise 2: Image Blending with OpenCV

Load two sample images from `scikit-image`, convert them from RGB to BGR color space, resize both to `640x420`, blend them using `cv2.addWeighted`, and display/save the outputs.

In [ ]:
import cv2
from skimage import data

# Load two sample RGB images
src1_rgb = data.chelsea()
src2_rgb = data.coffee()

# Convert RGB to BGR for OpenCV
img_a = cv2.cvtColor(src1_rgb, cv2.COLOR_RGB2BGR)
img_b = cv2.cvtColor(src2_rgb, cv2.COLOR_RGB2BGR)

# Resize both images to identical dimensions
width, height = 640, 420
img_a = cv2.resize(img_a, (width, height), interpolation=cv2.INTER_AREA)
img_b = cv2.resize(img_b, (width, height), interpolation=cv2.INTER_AREA)

# Weighted blending: 55% image 1 + 45% image 2
mixed_img = cv2.addWeighted(img_a, 0.55, img_b, 0.45, 0)

# Save inputs and blended result
path_a = os.path.join(OUTPUT_DIR, "Ex2_Image_A.jpg")
path_b = os.path.join(OUTPUT_DIR, "Ex2_Image_B.jpg")
path_mix = os.path.join(OUTPUT_DIR, "Ex2_Blended_Output.jpg")

cv2.imwrite(path_a, img_a)
cv2.imwrite(path_b, img_b)
cv2.imwrite(path_mix, mixed_img)

# Convert BGR back to RGB for Matplotlib visualization
fig, axes = plt.subplots(1, 3, figsize=(15, 4.8))
panels = [
    (img_a, "Image 1 (Chelsea)"),
    (img_b, "Image 2 (Coffee)"),
    (mixed_img, "Blended Result (0.55 / 0.45)"),
]
for ax, (frame, label) in zip(axes, panels):
    ax.imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
    ax.set_title(label)
    ax.axis("off")

plt.tight_layout()
plt.show()

print("Saved files:", [os.path.basename(p) for p in (path_a, path_b, path_mix)])

# Exercise 3: Prompt Engineering for Video Generation

- **Selected Tool:** Google Veo  
- **Duration & Aspect Ratio:** 3 seconds, 16:9 landscape  
- **Camera & Transition Instructions:** Smooth, steady slow push-in toward the subject; keep subject centered with natural depth of field; no abrupt cuts, camera shake, or text overlays.  

**Exact Prompt:**  
> A close-up, natural-light shot of a freshly poured cup of masala chai in a terracotta kulhad resting on a weathered teakwood table beside a small brass spoon. Gentle wisps of steam rise from the hot tea while the camera executes a smooth, slow push-in, highlighting the warm amber color and rich texture of the brew. Soft morning sunlight filters from the side, casting delicate shadows across the table. Realistic 4K cinematic footage, clean composition without any text, watermarks, or extra objects.

# Exercise 4: E-Commerce Text & Quantitative Analytics

Using the Flipkart Product Customer Reviews dataset from Kaggle:
1. Load and clean product names, prices, ratings, and review summaries.
2. Compute product-wise mean rating and mean price into a summary DataFrame (`items`).
3. Generate and display a Word Cloud from the `Summary` text column.

In [ ]:
import kagglehub
from wordcloud import WordCloud, STOPWORDS

# 1. Download and locate dataset CSV
dataset_dir = kagglehub.dataset_download("niraliivaghani/flipkart-product-customer-reviews-dataset")
csv_candidates = glob.glob(os.path.join(dataset_dir, "**", "*.csv"), recursive=True)
if not csv_candidates:
    raise FileNotFoundError("No CSV file found in the downloaded Kaggle dataset.")

target_csv = next(
    (p for p in csv_candidates if os.path.basename(p).lower() == "dataset-sa.csv"),
    csv_candidates[0]
)
reviews_raw = pd.read_csv(target_csv)
print(f"Loaded {os.path.basename(target_csv)} -> Shape: {reviews_raw.shape}")

# 2. Clean columns and filter invalid rows
reviews = reviews_raw.copy()
reviews.columns = reviews.columns.str.strip()

needed_cols = ["product_name", "product_price", "Rate", "Summary"]
missing = [c for c in needed_cols if c not in reviews.columns]
if missing:
    raise ValueError(f"Missing expected columns: {missing}")

def parse_numeric_col(series):
    cleaned = series.astype(str).str.replace(",", "", regex=False)
    extracted = cleaned.str.extract(r"(-?\d+\.?\d*)")[0]
    return pd.to_numeric(extracted, errors="coerce")

reviews["Rate"] = parse_numeric_col(reviews["Rate"])
reviews["product_price"] = parse_numeric_col(reviews["product_price"])
reviews["product_name"] = reviews["product_name"].astype("string").str.strip()

reviews = reviews.dropna(subset=["product_name", "Rate", "product_price"]).copy()
reviews = reviews[
    (reviews["Rate"].between(1, 5))
    & (reviews["product_price"] >= 0)
    & (reviews["product_name"] != "")
].copy()

print(f"Cleaned dataset shape: {reviews.shape}")
display(reviews[needed_cols].head())

# 3. Aggregate mean rating and mean price per product into `items`
items = (
    reviews.groupby("product_name", as_index=False)
    .agg(Mean_Rating=("Rate", "mean"), Mean_Price=("product_price", "mean"))
    .round({"Mean_Rating": 2, "Mean_Price": 2})
)

print(f"Total unique products: {len(items)}")
display(items.head(10))

# 4. Generate and display Word Cloud from Summary column
corpus = " ".join(reviews["Summary"].dropna().astype(str))
corpus = re.sub(r"\s+", " ", corpus).strip()
if not corpus:
    raise ValueError("No valid text available in Summary column.")

wc = WordCloud(
    width=1200,
    height=600,
    background_color="white",
    stopwords=set(STOPWORDS),
    max_words=180,
    collocations=False,
    colormap="plasma",
).generate(corpus)

plt.figure(figsize=(14, 7))
plt.imshow(wc, interpolation="bilinear")
plt.axis("off")
plt.title("Customer Review Summary — Word Cloud", fontsize=16)
plt.tight_layout()
plt.show()

# Save outputs
wc_path = os.path.join(OUTPUT_DIR, "Ex4_Reviews_WordCloud.png")
summary_xlsx_path = os.path.join(OUTPUT_DIR, "Ex4_Product_Summary.xlsx")
wc.to_file(wc_path)
items.to_excel(summary_xlsx_path, index=False)
print("Saved files:", os.path.abspath(wc_path), "and", os.path.abspath(summary_xlsx_path))

# Trigger download automatically if running inside Google Colab
try:
    from google.colab import files
    files.download(summary_xlsx_path)
    files.download(wc_path)
except ImportError:
    pass

# Exercise 5: Interactive Chatbot Prototype with Gradio

A lightweight rule-based Q&A assistant built using `gradio.ChatInterface`. It runs offline without external API keys and includes clickable sample questions.

In [ ]:
import gradio as gr

def gota_lab_assistant(message, history):
    query = message.lower().strip()

    if any(greet in query for greet in ["hello", "hi", "namaste", "hey"]):
        return "Namaste! Welcome to the GOT-A AI Lab Assistant. Ask me about Pandas, OpenCV, Word Clouds, or Gradio."
    elif "pandas" in query or "dataframe" in query:
        return "Pandas provides DataFrames for tabular data manipulation, cleaning, grouping, and exporting reports to Excel or CSV."
    elif "opencv" in query or "blend" in query or "image" in query:
        return "OpenCV processes images as NumPy arrays (in BGR format) and supports weighted blending via cv2.addWeighted()."
    elif "word cloud" in query or "wordcloud" in query or "summary" in query:
        return "A Word Cloud highlights high-frequency terms in text after removing null entries and common stopwords."
    elif "gradio" in query or "interface" in query:
        return "Gradio wraps Python functions in an interactive web UI, making it quick to test and share ML/AI prototypes."
    else:
        return (
            "I can help with concepts covered in Exercises 1–5 (Pandas, OpenCV image blending, "
            "Word Clouds, and Gradio). Please try one of the example questions below."
        )

chat_app = gr.ChatInterface(
    fn=gota_lab_assistant,
    title="GOT-A AI Lab Assistant",
    description="Interactive Q&A prototype for the NIELIT GOT-A practical exercises.",
    examples=[
        "How does Pandas help in data analysis?",
        "How does OpenCV blend two images?"
    ],
    chatbot=gr.Chatbot(height=380),
    textbox=gr.Textbox(placeholder="Ask a question about the lab exercises...", container=False, scale=7)
)

chat_app.launch(share=True)